In [0]:
# Importando bibliotecas
from pyspark.sql.functions import lower, col, when
from pyspark.sql.types import BooleanType

In [0]:
# Recebendo Table Bronze
df_bronze = spark.table (
    "projeto_profissional_genz.genz.genz_social_media_bronze"
)

In [0]:
df_bronze.printSchema()

In [0]:
# Recebendo em tabela silver colunas selecionadas da tabela bronze
df_silver = df_bronze.select (
    "daily_usage_hours",
    "num_platforms_used",
    "gender",
    "country",
    "age",
    "avg_session_minutes",
    "night_usage"
)

df_silver.show(10, truncate= False)

In [0]:
# Removendo Linhas em branco
df_silver = df_silver.dropna(
    subset=[
        "daily_usage_hours",
        "num_platforms_used",
        "gender",
        "country",
        "age",
        "avg_session_minutes",
        "night_usage"
    ]
)

df_silver.show(10, truncate=False)

In [0]:
# Removendo linhas duplicadas
df_silver = df_silver.drop_duplicates(
    subset=[
        "daily_usage_hours",
        "num_platforms_used",
        "gender",
        "country",
        "age",
        "avg_session_minutes",
        "night_usage"
    ]
)

df_silver.show(10, truncate=False)

In [0]:
# Padronizando colunas para português
df_silver = df_silver.withColumnRenamed("daily_usage_hours", "uso_diario")
df_silver = df_silver.withColumnRenamed("num_platforms_used", "num_plataformas_usadas")
df_silver = df_silver.withColumnRenamed("gender", "genero")
df_silver = df_silver.withColumnRenamed("country", "pais")
df_silver = df_silver.withColumnRenamed("age", "idade")
df_silver = df_silver.withColumnRenamed("avg_session_minutes", "tempo_medio_sessao")
df_silver = df_silver.withColumnRenamed("night_usage", "uso_noturno")

df_silver.show(10, truncate=False)

In [0]:
# Padronização de valores das colunas
df_silver = df_silver.withColumn(
    "genero", lower(col("genero"))
)

df_silver.show(5, truncate=False)

In [0]:
#Transformando coluna KM de `,` para `.` e transformando em Double!
df_silver = df_silver.withColumn(
    "uso_noturno",
    col("uso_noturno").cast(BooleanType())
)

df_silver.printSchema()
df_silver.show(80, truncate=False)

In [0]:
df_silver.printSchema()

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("projeto_profissional_genz.genz.genz_social_media_silver")

In [0]:
df = spark.table("projeto_profissional_genz.genz.genz_social_media_silver")
df.show()

In [0]:
%sql
select * from projeto_profissional_genz.genz.genz_social_media_silver